# ISY0101 — Evaluación Parcial N°3
## Implementación de Observabilidad — Agente Ripley Chile

Este notebook implementa:
- **Métricas de observabilidad**: latencia, precisión, consistencia, frecuencia de errores
- **Sistema de logs**: registro detallado de cada ejecución del agente
- **Análisis de trazabilidad**: identificación de patrones y cuellos de botella
- **Dashboard Streamlit**: visualización interactiva de métricas
- **Protocolos de seguridad**: sanitización de inputs y manejo de datos sensibles

> Requiere: GROQ_API_KEY (gratuita en console.groq.com)

## Celda 1 — Instalación

In [1]:
!pip install -q langchain==0.2.16 langchain-groq langchain-community langchain-text-splitters langchain-core faiss-cpu tiktoken sentence-transformers==2.7.0 huggingface_hub==0.23.4 streamlit pyngrok plotly pandas
print('Ve a Entorno de ejecución → Reiniciar entorno antes de continuar')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Celda 2 — Credenciales

In [1]:
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass('GROQ_API_KEY: ')
print('✅ Credencial guardada')

GROQ_API_KEY: ··········
✅ Credencial guardada


## Celda 3 — Crear datos simulados

In [2]:
import os, csv
os.makedirs('data', exist_ok=True)
os.makedirs('logs', exist_ok=True)

with open('data/politicas_garantia.txt','w',encoding='utf-8') as f:
    f.write("""POLÍTICAS DE GARANTÍA Y DEVOLUCIÓN — RIPLEY CHILE
1. GARANTÍA LEGAL: Todo producto tiene garantía mínima de 3 meses. Tecnología: 6 meses.
2. PROCESO: Presentar boleta en tienda. Plazo respuesta: 5 días hábiles.
3. DEVOLUCIONES: 10 días hábiles desde la compra, embalaje original.
4. DESPACHO: Si no llega en plazo, reembolso del despacho. Tras 5 días extra, cancelación total.
5. COBROS INCORRECTOS: Reversión en 5 días hábiles con comprobante.""")

with open('data/faq_atencion.txt','w',encoding='utf-8') as f:
    f.write("""PREGUNTAS FRECUENTES — RIPLEY
P: ¿Cuánto tiempo para devolución? R: 10 días hábiles.
P: ¿Producto dañado? R: Contactar en 24 horas.
P: ¿Garantía pantalla dañada sola? R: Sí, 6 meses si no es culpa del cliente.
P: ¿Devolución dinero vs reparación? R: Sí, según Ley 19.496.
P: ¿Sin solución de Ripley? R: Ir al SERNAC.""")

with open('data/ley_19496.txt','w',encoding='utf-8') as f:
    f.write("""LEY 19.496 — PROTECCIÓN CONSUMIDORES
Art. 20: El consumidor puede optar por reparación, reposición o devolución.
Plazo: 3 meses desde la compra.
Art. 23: Infracción por negligencia en calidad del bien o servicio.
Art. 58: SERNAC vela por cumplimiento. Proveedor responde en 10 días hábiles.""")

reclamos = [
    ['id','fecha','tipo','descripcion','resolucion','dias_resolucion'],
    ['001','2024-01-10','Garantia','TV dañado sin golpes','Cambio aceptado',3],
    ['002','2024-01-12','Despacho','Pedido no llegó','Reembolso despacho',2],
    ['003','2024-01-15','Devolucion','Talla incorrecta','Cambio por talla correcta',4],
    ['004','2024-01-18','Garantia','Celular no enciende','Cambio inmediato',1],
    ['005','2024-01-20','Cobro','Cobro duplicado','Reversión en 5 días',5],
    ['006','2024-01-22','Despacho','Producto equivocado','Retiro y reenvío',3],
    ['007','2024-01-25','Garantia','Lavadora con ruido','Reparación sin costo',7],
    ['008','2024-01-28','Devolucion','Embalaje dañado','Devolución completa',2],
    ['009','2024-02-01','Garantia','Refrigerador no enfría','Cambio equivalente',2],
    ['010','2024-02-05','SERNAC','Sin respuesta 15 días','Escalado SERNAC',12],
]
with open('data/reclamos_historicos.csv','w',newline='',encoding='utf-8') as f:
    csv.writer(f).writerows(reclamos)

print('✅ Datos creados')

✅ Datos creados


## Celda 4 — Indexación FAISS

In [3]:
from langchain_community.document_loaders import TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

documentos = []
for nombre, tipo in [('politicas_garantia.txt','interno'),
                     ('faq_atencion.txt','interno'),
                     ('ley_19496.txt','externo')]:
    docs = TextLoader(f'data/{nombre}', encoding='utf-8').load()
    for d in docs:
        d.metadata['source'] = nombre
        d.metadata['tipo']   = tipo
    documentos.extend(docs)

docs = CSVLoader('data/reclamos_historicos.csv', encoding='utf-8').load()
for d in docs:
    d.metadata['source'] = 'reclamos_historicos.csv'
    d.metadata['tipo']   = 'interno'
documentos.extend(docs)

chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(documentos)
print(f'Chunks: {len(chunks)}')

print('Cargando embeddings...')
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2'
)
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local('faiss_index')
print('✅ FAISS listo')

Chunks: 13
Cargando embeddings...


/tmp/ipykernel_2365/4095548775.py:27: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS listo


## Celda 5 — Herramientas del agente

In [4]:
from langchain_core.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import csv

from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2'
)
vector_store = FAISS.load_local('faiss_index', embeddings, allow_dangerous_deserialization=True)

@tool
def buscar_docs(consulta: str) -> str:
    """Busca información en documentos internos de Ripley y Ley 19.496."""
    docs = vector_store.similarity_search(consulta, k=4)
    return '\n\n'.join([f"[Fuente: {d.metadata.get('source')}]\n{d.page_content}" for d in docs])

@tool
def consultar_historial(tipo_reclamo: str) -> str:
    """Consulta historial de reclamos resueltos. Tipos: Garantia, Despacho, Devolucion, Cobro, SERNAC"""
    resultados = []
    with open('data/reclamos_historicos.csv', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if tipo_reclamo.lower() in row.get('tipo','').lower():
                resultados.append(f"ID:{row['id']} | {row['descripcion']} → {row['resolucion']} ({row['dias_resolucion']} días)")
    return f"{len(resultados)} casos similares:\n" + '\n'.join(resultados) if resultados else 'Sin casos previos'

@tool
def clasificar_reclamo(descripcion: str) -> str:
    """Clasifica el tipo de reclamo: Garantia, Despacho, Devolucion, Cobro, Otro."""
    d = descripcion.lower()
    if any(w in d for w in ['garantía','garantia','defecto','dañado','no funciona','no enciende','no enfría']): return 'Garantia'
    elif any(w in d for w in ['despacho','llegó','llego','entrega','envío','retraso']): return 'Despacho'
    elif any(w in d for w in ['devolucion','devolución','cambio','talla','devolver']): return 'Devolucion'
    elif any(w in d for w in ['cobro','cargo','duplicado','cobrado','tarjeta']): return 'Cobro'
    return 'Otro'

@tool
def evaluar_sernac(contexto: str) -> str:
    """Evalúa si el reclamo debe escalarse al SERNAC."""
    c = contexto.lower()
    señales = [('semanas sin respuesta','Sin respuesta >10 días hábiles — Art.58'),
               ('sin solución','Incumplimiento reiterado'),
               ('niegan garantía','Rechazo injustificado de garantía'),
               ('no han resuelto','Sin resolución tras plazo legal'),
               ('15 días','Plazo legal superado')]
    for señal, razon in señales:
        if señal in c: return f'SI — {razon}'
    return 'NO — Resoluble internamente'

herramientas = [buscar_docs, consultar_historial, clasificar_reclamo, evaluar_sernac]
print('✅ 4 herramientas listas')

✅ 4 herramientas listas


## Celda 6 — Agente con sistema de observabilidad
Instrumentamos el agente para capturar métricas en cada ejecución.

In [5]:
from langchain_groq import ChatGroq
from langchain.agents import AgentExecutor, create_react_agent
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
import time, json, csv, os
from datetime import datetime

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0,
               api_key=os.environ['GROQ_API_KEY'])

memoria = ConversationBufferMemory(memory_key='chat_history', return_messages=False)

react_prompt = PromptTemplate.from_template("""
Eres un agente especializado en reclamos postventa de Ripley Chile.
Herramientas disponibles: {tools}
Historial: {chat_history}

Formato OBLIGATORIO:
Question: {input}
Thought: razona qué herramienta usar
Action: nombre herramienta ({tool_names})
Action Input: input para la herramienta
Observation: resultado
Thought: tengo suficiente información
Final Answer: JSON con clasificacion, respuesta_sugerida, escalar_sernac, justificacion_escalamiento, herramientas_usadas, fuentes_usadas

Begin!
Question: {input}
Thought: {agent_scratchpad}
""")

agente = create_react_agent(llm=llm, tools=herramientas, prompt=react_prompt)
executor = AgentExecutor(agent=agente, tools=herramientas, memory=memoria,
                         verbose=False, max_iterations=6, handle_parsing_errors=True)

# ─── SISTEMA DE OBSERVABILIDAD ───
LOG_FILE = 'logs/metricas_agente.csv'

# Inicializar CSV de logs
with open(LOG_FILE, 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerow([
        'timestamp', 'id_ejecucion', 'reclamo', 'clasificacion',
        'latencia_seg', 'herramientas_usadas', 'num_herramientas',
        'escalar_sernac', 'exito', 'error', 'tokens_estimados'
    ])

contador_ejecucion = [0]

def ejecutar_con_metricas(reclamo: str, verbose: bool = True) -> dict:
    """Ejecuta el agente y registra métricas de observabilidad."""
    contador_ejecucion[0] += 1
    id_exec = f"EXEC-{contador_ejecucion[0]:03d}"
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # ── Protocolo de seguridad: sanitizar input ──
    reclamo_sanitizado = reclamo.strip()[:1000]  # max 1000 chars
    reclamo_sanitizado = reclamo_sanitizado.replace('<', '').replace('>', '')  # evitar injection

    inicio = time.time()
    exito = True
    error_msg = ''
    resultado = {}
    clasificacion = 'Desconocido'
    herramientas_list = []
    escalar = False

    try:
        raw = executor.invoke({'input': reclamo_sanitizado})
        output = raw['output'].strip()

        # Extraer JSON de la respuesta
        start = output.find('{')
        end   = output.rfind('}') + 1
        if start >= 0 and end > start:
            resultado = json.loads(output[start:end])
            clasificacion  = resultado.get('clasificacion', 'Desconocido')
            herramientas_list = resultado.get('herramientas_usadas', [])
            escalar        = resultado.get('escalar_sernac', False)
        else:
            exito = False
            error_msg = 'JSON no encontrado en respuesta'

    except Exception as e:
        exito = False
        error_msg = str(e)[:200]

    latencia = round(time.time() - inicio, 2)
    tokens_est = len(reclamo_sanitizado.split()) * 4  # estimación simple

    # Guardar en log CSV
    with open(LOG_FILE, 'a', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow([
            timestamp, id_exec, reclamo_sanitizado[:80], clasificacion,
            latencia, str(herramientas_list), len(herramientas_list),
            escalar, exito, error_msg, tokens_est
        ])

    if verbose:
        estado = '✅' if exito else '❌'
        print(f"{estado} [{id_exec}] Latencia: {latencia}s | Clasificación: {clasificacion} | SERNAC: {escalar} | Herramientas: {herramientas_list}")

    return {'id': id_exec, 'resultado': resultado, 'latencia': latencia,
            'exito': exito, 'error': error_msg}

print('✅ Agente con observabilidad listo')
print(f'   Logs se guardarán en: {LOG_FILE}')

✅ Agente con observabilidad listo
   Logs se guardarán en: logs/metricas_agente.csv


## Celda 7 — Ejecutar batería de pruebas
Corremos 8 reclamos variados para generar datos de observabilidad.

In [6]:
casos_prueba = [
    # (reclamo, clasificacion_esperada)
    ('Compré un televisor hace 3 semanas y la pantalla se dañó sola. Niegan la garantía.', 'Garantia'),
    ('Mi pedido lleva 10 días de retraso y no me dan información del despacho.', 'Despacho'),
    ('Me llegaron zapatos de talla incorrecta. Quiero cambiarlos sin pagar despacho.', 'Devolucion'),
    ('Me cobraron dos veces el mismo producto en mi tarjeta Ripley.', 'Cobro'),
    ('Llevo 3 semanas sin respuesta sobre mi refrigerador defectuoso. Voy al SERNAC.', 'Garantia'),
    ('Mi celular no enciende desde el día siguiente a la compra.', 'Garantia'),
    ('Compré online y llegó un producto completamente diferente al pedido.', 'Devolucion'),
    ('Necesito saber cuánto tiempo tengo para devolver un producto.', 'Otro'),
]

print('BATERÍA DE PRUEBAS — OBSERVABILIDAD')
print('=' * 70)

resultados_prueba = []
for reclamo, esperado in casos_prueba:
    r = ejecutar_con_metricas(reclamo)
    clasificacion_obtenida = r['resultado'].get('clasificacion', 'Error') if r['exito'] else 'Error'
    correcto = esperado.lower() in clasificacion_obtenida.lower()
    resultados_prueba.append({
        'reclamo': reclamo[:60],
        'esperado': esperado,
        'obtenido': clasificacion_obtenida,
        'correcto': correcto,
        'latencia': r['latencia'],
        'exito': r['exito']
    })

# Calcular métricas globales
total = len(resultados_prueba)
correctos = sum(1 for r in resultados_prueba if r['correcto'])
exitosos   = sum(1 for r in resultados_prueba if r['exito'])
latencias  = [r['latencia'] for r in resultados_prueba]

print(f'\n═══ MÉTRICAS GLOBALES ═══')
print(f'Precisión          : {correctos}/{total} = {correctos/total*100:.1f}%')
print(f'Tasa de éxito      : {exitosos}/{total} = {exitosos/total*100:.1f}%')
print(f'Latencia promedio  : {sum(latencias)/len(latencias):.2f}s')
print(f'Latencia máxima    : {max(latencias):.2f}s')
print(f'Latencia mínima    : {min(latencias):.2f}s')
print(f'Frecuencia errores : {(total-exitosos)/total*100:.1f}%')

BATERÍA DE PRUEBAS — OBSERVABILIDAD
✅ [EXEC-001] Latencia: 9.39s | Clasificación: Garantia | SERNAC: NO | Herramientas: ['clasificar_reclamo', 'buscar_docs', 'consultar_historial', 'evaluar_sernac']
✅ [EXEC-002] Latencia: 16.14s | Clasificación: Despacho | SERNAC: NO | Herramientas: ['clasificar_reclamo', 'buscar_docs', 'evaluar_sernac', 'consultar_historial']
❌ [EXEC-003] Latencia: 9.81s | Clasificación: Desconocido | SERNAC: False | Herramientas: []
✅ [EXEC-004] Latencia: 68.12s | Clasificación: Cobro | SERNAC: NO | Herramientas: ['clasificar_reclamo', 'buscar_docs', 'evaluar_sernac']
❌ [EXEC-005] Latencia: 0.05s | Clasificación: Desconocido | SERNAC: False | Herramientas: []
❌ [EXEC-006] Latencia: 0.05s | Clasificación: Desconocido | SERNAC: False | Herramientas: []
❌ [EXEC-007] Latencia: 0.05s | Clasificación: Desconocido | SERNAC: False | Herramientas: []
❌ [EXEC-008] Latencia: 0.05s | Clasificación: Desconocido | SERNAC: False | Herramientas: []

═══ MÉTRICAS GLOBALES ═══
Precisi

## Celda 8 — Análisis de logs y trazabilidad

In [7]:
import pandas as pd

# Cargar logs
df = pd.read_csv('logs/metricas_agente.csv')
print('ANÁLISIS DE LOGS Y TRAZABILIDAD')
print('=' * 60)
print(f'\nTotal ejecuciones registradas: {len(df)}')
print(f'\nPrimeras 5 entradas del log:')
print(df[['id_ejecucion','clasificacion','latencia_seg','exito','escalar_sernac']].to_string())

print(f'\n─── ANÁLISIS DE LATENCIA ───')
print(f'Promedio : {df["latencia_seg"].mean():.2f}s')
print(f'Máxima   : {df["latencia_seg"].max():.2f}s')
print(f'Mínima   : {df["latencia_seg"].min():.2f}s')
print(f'Desv. std: {df["latencia_seg"].std():.2f}s')

# Identificar cuellos de botella (latencia > promedio + 1 std)
umbral = df['latencia_seg'].mean() + df['latencia_seg'].std()
cuellos = df[df['latencia_seg'] > umbral]
print(f'\n─── CUELLOS DE BOTELLA (latencia > {umbral:.2f}s) ───')
if len(cuellos) > 0:
    for _, row in cuellos.iterrows():
        print(f'  [{row["id_ejecucion"]}] {row["latencia_seg"]}s — {row["clasificacion"]} — {row["reclamo"][:50]}...')
else:
    print('  No se detectaron cuellos de botella')

print(f'\n─── DISTRIBUCIÓN POR CLASIFICACIÓN ───')
print(df['clasificacion'].value_counts().to_string())

print(f'\n─── CASOS QUE ESCALAN A SERNAC ───')
sernac = df[df['escalar_sernac'] == True]
print(f'Total: {len(sernac)} de {len(df)} ({len(sernac)/len(df)*100:.1f}%)')

print(f'\n─── ERRORES DETECTADOS ───')
errores = df[df['exito'] == False]
if len(errores) > 0:
    for _, row in errores.iterrows():
        print(f'  [{row["id_ejecucion"]}] Error: {row["error"]}')
else:
    print('  No se detectaron errores en esta sesión')

print(f'\n─── PATRONES Y ANOMALÍAS ───')
print(f'Ejecuciones exitosas     : {df["exito"].sum()} ({df["exito"].mean()*100:.1f}%)')
print(f'Promedio herramientas/call: {df["num_herramientas"].mean():.1f}')
print(f'Tokens promedio estimados : {df["tokens_estimados"].mean():.0f}')

ANÁLISIS DE LOGS Y TRAZABILIDAD

Total ejecuciones registradas: 8

Primeras 5 entradas del log:
  id_ejecucion clasificacion  latencia_seg  exito escalar_sernac
0     EXEC-001      Garantia          9.39   True             NO
1     EXEC-002      Despacho         16.14   True             NO
2     EXEC-003   Desconocido          9.81  False          False
3     EXEC-004         Cobro         68.12   True             NO
4     EXEC-005   Desconocido          0.05  False          False
5     EXEC-006   Desconocido          0.05  False          False
6     EXEC-007   Desconocido          0.05  False          False
7     EXEC-008   Desconocido          0.05  False          False

─── ANÁLISIS DE LATENCIA ───
Promedio : 12.96s
Máxima   : 68.12s
Mínima   : 0.05s
Desv. std: 23.12s

─── CUELLOS DE BOTELLA (latencia > 36.08s) ───
  [EXEC-004] 68.12s — Cobro — Me cobraron dos veces el mismo producto en mi tarj...

─── DISTRIBUCIÓN POR CLASIFICACIÓN ───
clasificacion
Desconocido    5
Garantia       

## Celda 9 — Protocolos de seguridad y uso responsable

In [8]:
print('PROTOCOLOS DE SEGURIDAD IMPLEMENTADOS')
print('=' * 60)

# ── 1. Sanitización de inputs ──
def sanitizar_input(texto: str) -> tuple:
    """Sanitiza el input del usuario para prevenir inyecciones y datos maliciosos."""
    alertas = []
    texto_limpio = texto.strip()

    # Límite de longitud
    if len(texto_limpio) > 1000:
        texto_limpio = texto_limpio[:1000]
        alertas.append('Input truncado a 1000 caracteres')

    # Remover caracteres peligrosos
    chars_peligrosos = ['<', '>', '{', '}', 'SELECT', 'DROP', 'INSERT', '--']
    for char in chars_peligrosos:
        if char in texto_limpio:
            texto_limpio = texto_limpio.replace(char, '')
            alertas.append(f'Carácter/patrón peligroso removido: {char}')

    return texto_limpio, alertas

# ── 2. Anonimización de datos sensibles en logs ──
def anonimizar_rut(texto: str) -> str:
    """Reemplaza RUTs en el texto para proteger datos personales."""
    import re
    patron_rut = r'\b\d{7,8}-[\dkK]\b'
    return re.sub(patron_rut, '[RUT-ANONIMIZADO]', texto)

# ── 3. Pruebas de seguridad ──
print('\n1. SANITIZACIÓN DE INPUTS:')
inputs_maliciosos = [
    'Mi reclamo es sobre el TV <script>alert(1)</script>',
    'SELECT * FROM reclamos WHERE id=1; DROP TABLE reclamos;',
    'A' * 1500,  # input muy largo
]
for inp in inputs_maliciosos:
    limpio, alertas = sanitizar_input(inp)
    print(f'  Input original : {inp[:50]}...')
    print(f'  Input limpio   : {limpio[:50]}...')
    print(f'  Alertas        : {alertas}\n')

print('2. ANONIMIZACIÓN DE DATOS SENSIBLES:')
texto_con_rut = 'El cliente con RUT 12.345.678-9 presenta reclamo por garantía'
print(f'  Original    : {texto_con_rut}')
print(f'  Anonimizado : {anonimizar_rut(texto_con_rut)}')

print('\n3. CRITERIOS ÉTICOS Y NORMATIVOS IMPLEMENTADOS:')
criterios = [
    '✅ Trazabilidad completa: cada respuesta cita sus fuentes documentales',
    '✅ No discriminación: el agente aplica la misma normativa a todos los casos',
    '✅ Transparencia: el agente indica cuando no tiene información suficiente',
    '✅ Privacidad por diseño: datos de clientes no se almacenan en logs completos',
    '✅ Cumplimiento normativo: respuestas alineadas con Ley 19.496 y SERNAC',
    '✅ Supervisión humana: el agente sugiere respuestas, no decide autónomamente',
]
for c in criterios:
    print(f'  {c}')

PROTOCOLOS DE SEGURIDAD IMPLEMENTADOS

1. SANITIZACIÓN DE INPUTS:
  Input original : Mi reclamo es sobre el TV <script>alert(1)</script...
  Input limpio   : Mi reclamo es sobre el TV scriptalert(1)/script...
  Alertas        : ['Carácter/patrón peligroso removido: <', 'Carácter/patrón peligroso removido: >']

  Input original : SELECT * FROM reclamos WHERE id=1; DROP TABLE recl...
  Input limpio   :  * FROM reclamos WHERE id=1;  TABLE reclamos;...
  Alertas        : ['Carácter/patrón peligroso removido: SELECT', 'Carácter/patrón peligroso removido: DROP']

  Input original : AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...
  Input limpio   : AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...
  Alertas        : ['Input truncado a 1000 caracteres']

2. ANONIMIZACIÓN DE DATOS SENSIBLES:
  Original    : El cliente con RUT 12.345.678-9 presenta reclamo por garantía
  Anonimizado : El cliente con RUT 12.345.678-9 presenta reclamo por garantía

3. CRITERIOS ÉTICOS Y NORMATIVOS IMPLEM

## Celda 10 — Dashboard Streamlit
Genera el archivo del dashboard y lo lanza con ngrok.

In [9]:
dashboard_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title='Observabilidad — Agente Ripley', layout='wide')
st.title('📊 Dashboard de Observabilidad — Agente de Reclamos Ripley Chile')
st.markdown('**ISY0101 — Evaluación Parcial N°3 | DuocUC 2025**')

try:
    df = pd.read_csv('logs/metricas_agente.csv')
except:
    st.error('No se encontró el archivo de logs.')
    st.stop()

# Convertir columnas a tipos correctos
df['exito'] = df['exito'].astype(str).str.strip().str.lower().map({'true': True, 'false': False, '1': True, '0': False}).fillna(False)
df['escalar_sernac'] = df['escalar_sernac'].astype(str).str.strip().str.lower().map({'true': True, 'false': False, 'si': True, 'no': False, '1': True, '0': False}).fillna(False)
df['latencia_seg'] = pd.to_numeric(df['latencia_seg'], errors='coerce').fillna(0)

# KPIs
st.header('Métricas Clave')
col1, col2, col3, col4 = st.columns(4)
col1.metric('Total Ejecuciones', len(df))
col2.metric('Tasa de Éxito', f"{df['exito'].mean()*100:.1f}%")
col3.metric('Latencia Promedio', f"{df['latencia_seg'].mean():.2f}s")
col4.metric('Latencia Máxima', f"{df['latencia_seg'].max():.2f}s")

st.divider()

col1, col2 = st.columns(2)
with col1:
    st.subheader('⏱ Latencia por Ejecución')
    fig = px.bar(df, x='id_ejecucion', y='latencia_seg',
                 color='exito', color_discrete_map={True: '#2E86AB', False: '#E84855'},
                 labels={'latencia_seg': 'Latencia (s)', 'id_ejecucion': 'Ejecución'})
    umbral = df['latencia_seg'].mean() + df['latencia_seg'].std()
    fig.add_hline(y=umbral, line_dash='dash', line_color='orange',
                  annotation_text=f'Umbral ({umbral:.1f}s)')
    fig.add_hline(y=df['latencia_seg'].mean(), line_dash='dot', line_color='green',
                  annotation_text=f'Promedio ({df["latencia_seg"].mean():.1f}s)')
    st.plotly_chart(fig, use_container_width=True)

with col2:
    st.subheader('📂 Distribución por Clasificación')
    dist = df['clasificacion'].value_counts().reset_index()
    dist.columns = ['Clasificación', 'Cantidad']
    fig2 = px.pie(dist, values='Cantidad', names='Clasificación',
                  color_discrete_sequence=px.colors.qualitative.Set2)
    st.plotly_chart(fig2, use_container_width=True)

st.divider()

col3, col4 = st.columns(2)
with col3:
    st.subheader('🔧 Uso de Herramientas por Ejecución')
    fig3 = px.bar(df, x='id_ejecucion', y='num_herramientas', color='clasificacion')
    st.plotly_chart(fig3, use_container_width=True)

with col4:
    st.subheader('📈 Latencia por Tipo de Reclamo')
    lat_tipo = df.groupby('clasificacion')['latencia_seg'].mean().reset_index()
    lat_tipo.columns = ['Clasificación', 'Promedio']
    fig4 = px.bar(lat_tipo, x='Clasificación', y='Promedio', color='Clasificación')
    st.plotly_chart(fig4, use_container_width=True)

st.divider()

st.subheader('📋 Registro de Ejecuciones')
st.dataframe(df[['timestamp','id_ejecucion','clasificacion','latencia_seg','num_herramientas','escalar_sernac','exito','error']], use_container_width=True)

st.subheader('⚠️ Anomalías Detectadas')
umbral = df["latencia_seg"].mean() + df["latencia_seg"].std()
anomalias = df[(df['latencia_seg'] > umbral) | (df['exito'] == False)]
if len(anomalias) > 0:
    st.warning(f"{len(anomalias)} anomalía(s) detectada(s)")
    st.dataframe(anomalias[['id_ejecucion','clasificacion','latencia_seg','exito','error']])
else:
    st.success('No se detectaron anomalías')

st.caption('ISY0101 — Ingeniería de Soluciones con IA | DuocUC 2025')
'''

with open('dashboard.py', 'w', encoding='utf-8') as f:
    f.write(dashboard_code)

print('✅ Dashboard actualizado')

✅ Dashboard actualizado


## Celda 11 — Lanzar el dashboard
Crearse cuenta en ngrok https://dashboard.ngrok.com/login

Abre el dashboard en el navegador via ngrok.

In [10]:
from pyngrok import ngrok
import subprocess, time

ngrok.kill()
time.sleep(2)

ngrok.set_auth_token("TU_TOKEN_NGROK_AQUI")

# Usar puerto diferente para evitar conflicto
proceso = subprocess.Popen(
    ['streamlit', 'run', 'dashboard.py', '--server.port=8502',
     '--server.headless=true', '--server.enableCORS=false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(4)

tunnel = ngrok.connect(8502)
url_publica = tunnel.public_url

print('✅ Dashboard lanzado')
print(f'🌐 URL pública: {url_publica}')
print('\n⚠️  Abre esa URL en tu navegador para ver el dashboard.')
print('    Toma capturas de pantalla para el informe.')

✅ Dashboard lanzado
🌐 URL pública: https://valid-appease-immerse.ngrok-free.dev

⚠️  Abre esa URL en tu navegador para ver el dashboard.
    Toma capturas de pantalla para el informe.


## Celda 12 — Propuesta de mejoras basada en métricas

In [11]:
import pandas as pd

df = pd.read_csv('logs/metricas_agente.csv')

df['exito'] = df['exito'].astype(str).str.strip().str.lower().map({'true': True, 'false': False, '1': True, '0': False}).fillna(False)
df['escalar_sernac'] = df['escalar_sernac'].astype(str).str.strip().str.lower().map({'true': True, 'false': False, 'si': True, 'no': False, '1': True, '0': False}).fillna(False)
df['latencia_seg'] = pd.to_numeric(df['latencia_seg'], errors='coerce').fillna(0)

print('PROPUESTA DE MEJORAS BASADA EN MÉTRICAS')
print('=' * 60)

latencia_prom = df['latencia_seg'].mean()
tasa_exito    = df['exito'].mean() * 100
tasa_sernac   = df['escalar_sernac'].mean() * 100
prom_tools    = df['num_herramientas'].mean()

print(f'\nResumen de métricas:')
print(f'  Latencia promedio : {latencia_prom:.2f}s')
print(f'  Tasa de éxito     : {tasa_exito:.1f}%')
print(f'  Casos SERNAC      : {tasa_sernac:.1f}%')
print(f'  Tools por llamada : {prom_tools:.1f}')

print(f'\nRECOMENDACIONES:')

if latencia_prom > 10:
    print('\n⚡ MEJORA 1 — Reducir latencia:')
    print('   Problema: latencia promedio supera 10s, impacta UX.')
    print('   Solución: implementar caché de respuestas frecuentes con Redis.')
    print('   Impacto esperado: reducción del 40% en latencia para consultas repetidas.')
else:
    print('\n✅ MEJORA 1 — Latencia aceptable, monitorear en producción con mayor carga.')

if tasa_exito < 100:
    print(f'\n🔧 MEJORA 2 — Reducir tasa de errores (actual: {100-tasa_exito:.1f}%):')
    print('   Problema: errores de parsing JSON en respuestas del LLM.')
    print('   Solución: implementar OutputParser estructurado de LangChain.')
    print('   Impacto esperado: eliminación de errores de formato.')
else:
    print('\n✅ MEJORA 2 — Tasa de éxito 100%, mantener monitoreo activo.')

if prom_tools > 3:
    print(f'\n🛠️  MEJORA 3 — Optimizar uso de herramientas (actual: {prom_tools:.1f}/llamada):')
    print('   Problema: el agente usa demasiadas herramientas por consulta.')
    print('   Solución: mejorar el prompt para guiar mejor la selección de herramientas.')
else:
    print(f'\n✅ MEJORA 3 — Uso de herramientas eficiente ({prom_tools:.1f}/llamada).')

print('\n📈 MEJORA 4 — Escalabilidad:')
print('   Implementar procesamiento en batch para reclamos de bajo riesgo.')
print('   Reservar llamadas síncronas al LLM para casos que requieren SERNAC.')

print('\n🔒 MEJORA 5 — Seguridad en producción:')
print('   Implementar rate limiting por IP para evitar abuso del agente.')
print('   Agregar autenticación OAuth2 para acceso al dashboard de monitoreo.')

PROPUESTA DE MEJORAS BASADA EN MÉTRICAS

Resumen de métricas:
  Latencia promedio : 12.96s
  Tasa de éxito     : 37.5%
  Casos SERNAC      : 0.0%
  Tools por llamada : 1.4

RECOMENDACIONES:

⚡ MEJORA 1 — Reducir latencia:
   Problema: latencia promedio supera 10s, impacta UX.
   Solución: implementar caché de respuestas frecuentes con Redis.
   Impacto esperado: reducción del 40% en latencia para consultas repetidas.

🔧 MEJORA 2 — Reducir tasa de errores (actual: 62.5%):
   Problema: errores de parsing JSON en respuestas del LLM.
   Solución: implementar OutputParser estructurado de LangChain.
   Impacto esperado: eliminación de errores de formato.

✅ MEJORA 3 — Uso de herramientas eficiente (1.4/llamada).

📈 MEJORA 4 — Escalabilidad:
   Implementar procesamiento en batch para reclamos de bajo riesgo.
   Reservar llamadas síncronas al LLM para casos que requieren SERNAC.

🔒 MEJORA 5 — Seguridad en producción:
   Implementar rate limiting por IP para evitar abuso del agente.
   Agregar 